## Verbalization

Since HotpotQA provides short-form answers, we apply an answer verbalization step to transform entity-based answers into complete natural language sentences, ensuring compatibility with LLM-based evaluation metrics such as RAGAs.

In [ ]:
import pandas as pd
import numpy as np
import os
import dotenv
import time
import google.generativeai as genai
import asyncio
import ipywidgets
from asyncio import Semaphore
from tqdm.auto import tqdm  # Auto-select best version (notebook or console)

In [ ]:
# load dataset
train = pd.read_csv("../data/hotpot_qa/train.csv", encoding='utf-8-sig')
valid = pd.read_csv("../data/hotpot_qa/valid.csv", encoding='utf-8-sig')

In [ ]:
train.head()

,id,question,answer,type,level,supporting_facts,context
0,5a7a06935542990198eaf050,Which magazine was started first Arthur's Maga...,Arthur's Magazine,comparison,medium,"{'title': [""Arthur's Magazine"", 'First for Wom...",{'title': ['Radio City (Indian radio station)'...
1,5a879ab05542996e4f30887e,The Oberoi family is part of a hotel company t...,Delhi,bridge,medium,"{'title': ['Oberoi family', 'The Oberoi Group'...","{'title': ['Ritz-Carlton Jakarta', 'Oberoi fam..."
2,5a8d7341554299441c6b9fe5,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,bridge,hard,"{'title': ['Allie Goertz', 'Allie Goertz', 'Al...","{'title': ['Lisa Simpson', 'Marge Simpson', 'B..."
3,5a82171f5542990a1d231f4a,What nationality was James Henry Miller's wife?,American,bridge,medium,"{'title': ['Peggy Seeger', 'Peggy Seeger', 'Ew...","{'title': ['Moloch: or, This Gentile World', '..."
4,5a84dd955542997b5ce3ff79,Cadmium Chloride is slightly soluble in this c...,alcohol,bridge,medium,"{'title': ['Cadmium chloride', 'Ethanol'], 'se...","{'title': ['Cadmium chloride', 'Water blue', '..."


In [ ]:
dotenv.load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
SLEEP_TIME = 0.2

In [ ]:
# List available models
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-p

In [ ]:
MODEL_NAME = "gemini-2.5-flash"  
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel(MODEL_NAME)

In [ ]:
PROMPT_TEMPLATE = """
You are a factual assistant.
Given a question and its short answer, rewrite the answer into
ONE complete, factual sentence that directly answers the question.

Rules:
- Do NOT add new facts.
- Do NOT explain beyond the answer.
- Keep it concise and precise.

Question: {question}
Short answer: {answer}
"""

In [ ]:
def verbalize(question, answer):
    prompt = PROMPT_TEMPLATE.format(
        question=question.strip(),
        answer=answer.strip()
    )
    response = model.generate_content(prompt)
    return response.text.strip()

In [ ]:
def verbalize_batch(df, checkpoint_path=None, checkpoint_freq=100):
    df = df.copy()  # Tránh mutate original
    verbalized_answers = []
    
    # Resume from checkpoint if exists
    start_idx = 0
    if checkpoint_path and os.path.exists(checkpoint_path):
        checkpoint_df = pd.read_csv(checkpoint_path)
        verbalized_answers = checkpoint_df['verbalized_answer'].tolist()
        start_idx = len(verbalized_answers)
        print(f"Resuming from index {start_idx}")
    
    # Progress bar with detailed info
    pbar = tqdm(total=len(df), initial=start_idx, 
                desc="Verbalizing answers", 
                unit="answer",
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]')
    
    for idx, row in enumerate(df.itertuples(), start=0):
        if idx < start_idx:
            continue
            
        try:
            verbalized_answer = verbalize(row.question, row.answer)
            pbar.set_postfix({"status": "✓", "last_length": len(verbalized_answer)})
        except Exception as e:
            print(f"\nError at index {idx}: {e}")
            # save checkpoint & exit
            if checkpoint_path:
                temp_df = df.iloc[:idx].copy()
                temp_df['verbalized_answer'] = verbalized_answers
                temp_df.to_csv(checkpoint_path, index=False, encoding='utf-8-sig')
                print(f"Checkpoint saved at index {idx}")
            break
        
        verbalized_answers.append(verbalized_answer)
        pbar.update(1)
        time.sleep(SLEEP_TIME)
        
        # Save checkpoint periodically
        if checkpoint_path and (idx + 1) % checkpoint_freq == 0:
            temp_df = df.iloc[:idx+1].copy()
            temp_df['verbalized_answer'] = verbalized_answers
            temp_df.to_csv(checkpoint_path, index=False, encoding='utf-8-sig')
            pbar.set_postfix({"status": "💾 checkpoint saved"})
    
    pbar.close()
    df['verbalized_answer'] = verbalized_answers
    return df

In [ ]:
# train2 = verbalize_batch(train)
# valid2 = verbalize_batch(valid)

## Async Version - Faster Processing

Sử dụng async để xử lý song song nhiều requests, giảm thời gian xử lý từ ~11 giờ xuống ~2-3 giờ cho 200K dòng.

In [ ]:
async def verbalize_async(question, answer, semaphore):
    """Async version with rate limiting"""
    async with semaphore:
        try:
            prompt = PROMPT_TEMPLATE.format(
                question=question.strip(),
                answer=answer.strip()
            )
            loop = asyncio.get_event_loop()
            response = await loop.run_in_executor(
                None, 
                lambda: model.generate_content(prompt)
            )
            return response.text.strip()
        except Exception as e:
            print(f"Error verbalizing answer: {e}")
            return answer  # Fallback to original

In [24]:
async def verbalize_batch_async(df, checkpoint_path=None, checkpoint_freq=500, max_concurrent=10):
    """
    Async batch processing - nhanh hơn 3-5 lần so với sync version
    
    Args:
        df: DataFrame to process
        checkpoint_path: Path to save checkpoints
        checkpoint_freq: Save checkpoint every N items
        max_concurrent: Max concurrent requests (10-15, default 10)
    """
    df = df.copy()
    
    # Resume from checkpoint
    start_idx = 0
    verbalized_answers = [None] * len(df)  # Pre-allocate full size
    if checkpoint_path and os.path.exists(checkpoint_path):
        checkpoint_df = pd.read_csv(checkpoint_path)
        checkpoint_answers = checkpoint_df['verbalized_answer'].tolist()
        start_idx = len(checkpoint_answers)
        # Copy checkpoint data to pre-allocated list
        verbalized_answers[:start_idx] = checkpoint_answers
        print(f"▶ Resuming from index {start_idx}")
    
    semaphore = Semaphore(max_concurrent)
    
    # Progress bar
    pbar = tqdm(total=len(df), initial=start_idx,
                desc=f"Async ({max_concurrent} concurrent)",
                unit="ans",
                position=0, leave=True)
    
    errors = []
    completed = [start_idx]  # Use list to allow modification in nested function
    
    async def process_row(idx, row):
        if idx < start_idx:
            return
        
        try:
            result = await verbalize_async(row.question, row.answer, semaphore)
            verbalized_answers[idx] = result
        except Exception as e:
            # save checkpoint & exit
            print(f"\nError at index {idx}: {e}", flush=True)
            if checkpoint_path:
                temp_df = df.iloc[:idx].copy()
                temp_df['verbalized_answer'] = verbalized_answers
                temp_df.to_csv(checkpoint_path, index=False, encoding='utf-8-sig')
                print(f"💾 Checkpoint saved at index {idx}")
                pbar.close()
            return
            
        
        pbar.update(1)
        completed[0] += 1
        
        # Print progress every 100 items
        if completed[0] % 100 == 0:
            print(f"✓ Progress: {completed[0]}/{len(df)} ({completed[0]/len(df)*100:.1f}%)", flush=True)
        
        # Save checkpoint
        if checkpoint_path and completed[0] % checkpoint_freq == 0:
            temp_df = df.iloc[:completed[0]].copy()
            temp_df['verbalized_answer'] = verbalized_answers[:completed[0]]
            temp_df.to_csv(checkpoint_path, index=False, encoding='utf-8-sig')
            print(f"💾 Checkpoint saved: {completed[0]}/{len(df)}", flush=True)
    
    # Create and run all tasks
    print(f"🚀 Starting async processing: {len(df)} items with {max_concurrent} concurrent requests...")
    tasks = [process_row(idx, row) for idx, row in enumerate(df.itertuples())]
    await asyncio.gather(*tasks)
    
    pbar.close()
    
    print(f"\n✅ Completed! Processed {len(df)} items")
    if errors:
        print(f"⚠ {len(errors)} errors occurred")
    
    df['verbalized_answer'] = verbalized_answers
    return df

In [25]:
# Test với 100 rows trước
test_sample = train.head(100).copy()
test_result = await verbalize_batch_async(
    test_sample, 
    checkpoint_path="../data/hotpot_qa/test_async.csv",
    max_concurrent=10
)

Async (10 concurrent):   0%|          | 0/100 [00:00<?, ?ans/s]

🚀 Starting async processing: 100 items with 10 concurrent requests...
✓ Progress: 100/100 (100.0%)

✅ Completed! Processed 100 items


In [26]:
# Xem kết quả test
test_result[['question', 'answer', 'verbalized_answer']].head()

,question,answer,verbalized_answer
0,Which magazine was started first Arthur's Maga...,Arthur's Magazine,Arthur's Magazine was started first.
1,The Oberoi family is part of a hotel company t...,Delhi,The Oberoi family is part of a hotel company t...
2,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,"Matt Groening named the ""The Simpsons"" charact..."
3,What nationality was James Henry Miller's wife?,American,James Henry Miller's wife was American.
4,Cadmium Chloride is slightly soluble in this c...,alcohol,The chemical in which Cadmium Chloride is slig...


In [10]:
# # keep 3837 first rows of valid_async_checkpoint.csv
# valid_checkpoint = pd.read_csv("../data/hotpot_qa/valid_async_checkpoint.csv")
# checkpoint = valid_checkpoint.head(3837)
# checkpoint.info()
# # clear original file
# os.remove("../data/hotpot_qa/valid_async_checkpoint.csv")
# # save checkpoint
# checkpoint.to_csv("../data/hotpot_qa/valid_async_checkpoint.csv", index=False, encoding='utf-8-sig')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3837 entries, 0 to 3836
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 3837 non-null   object
 1   question           3837 non-null   object
 2   answer             3837 non-null   object
 3   type               3837 non-null   object
 4   level              3837 non-null   object
 5   supporting_facts   3837 non-null   object
 6   context            3837 non-null   object
 7   verbalized_answer  3831 non-null   object
dtypes: object(8)
memory usage: 239.9+ KB


In [27]:
valid_async = await verbalize_batch_async(
    valid,
    checkpoint_path="../data/hotpot_qa/valid_async_checkpoint.csv",
    checkpoint_freq=500,
    max_concurrent=12
)
valid_async.to_csv("../data/hotpot_qa/valid_async.csv", index=False, encoding='utf-8-sig')

▶ Resuming from index 4000


Async (12 concurrent):  54%|#####4    | 4000/7405 [00:00<?, ?ans/s]

🚀 Starting async processing: 7405 items with 12 concurrent requests...
✓ Progress: 4100/7405 (55.4%)
✓ Progress: 4200/7405 (56.7%)
✓ Progress: 4300/7405 (58.1%)
✓ Progress: 4400/7405 (59.4%)
✓ Progress: 4500/7405 (60.8%)
💾 Checkpoint saved: 4500/7405
✓ Progress: 4600/7405 (62.1%)
✓ Progress: 4700/7405 (63.5%)
✓ Progress: 4800/7405 (64.8%)
✓ Progress: 4900/7405 (66.2%)
✓ Progress: 5000/7405 (67.5%)
💾 Checkpoint saved: 5000/7405
✓ Progress: 5100/7405 (68.9%)
✓ Progress: 5200/7405 (70.2%)
✓ Progress: 5300/7405 (71.6%)
✓ Progress: 5400/7405 (72.9%)
✓ Progress: 5500/7405 (74.3%)
💾 Checkpoint saved: 5500/7405
✓ Progress: 5600/7405 (75.6%)
✓ Progress: 5700/7405 (77.0%)
✓ Progress: 5800/7405 (78.3%)
✓ Progress: 5900/7405 (79.7%)
✓ Progress: 6000/7405 (81.0%)
💾 Checkpoint saved: 6000/7405
✓ Progress: 6100/7405 (82.4%)
✓ Progress: 6200/7405 (83.7%)
✓ Progress: 6300/7405 (85.1%)
✓ Progress: 6400/7405 (86.4%)
✓ Progress: 6500/7405 (87.8%)
💾 Checkpoint saved: 6500/7405
✓ Progress: 6600/7405 (89.1%)

In [ ]:
train_async = await verbalize_batch_async(
    train, 
    checkpoint_path="../data/hotpot_qa/train_async_checkpoint.csv",
    checkpoint_freq=1000,
    max_concurrent=12 # Tăng lên nếu không bị rate limit
)
train_async.to_csv("../data/hotpot_qa/train_async.csv", index=False, encoding='utf-8-sig')